# U-Net comparison run on Colab

Trains one U-Net comparison run on a free Colab GPU and saves it to Google Drive. Everything matches the
main run (`unet_28spk`) except the one setting the comparison changes:

| Run | Changes | Question it answers |
|---|---|---|
| `unet_28spk_psm` | `--target psm` | How much does reusing the noisy phase cost? |
| `unet_28spk_c8` | `--base-channels 8` | Does the gain come from the design or from model size? |

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. The repository is private, so Colab needs a GitHub token to clone it. On github.com: Settings →
   Developer settings → Personal access tokens → **Fine-grained tokens** → Generate new token. Under
   *Repository access* pick only `noisy_speech`; under *Permissions*, **Contents: Read-only**.
3. In Colab, open the **🔑 Secrets** panel (left sidebar), add a secret named `GITHUB_TOKEN` with the
   token as its value, and switch on **Notebook access**. The token never appears in the notebook or its output.

Then run the cells in order.

**If Colab disconnects:** run every cell again from the top. The data is downloaded again, and training
continues from the last finished epoch saved on Drive.

**When it finishes:** download `MyDrive/noisy_speech_runs/<run name>/` from Google Drive into
`unet_model/outputs/<run name>/` on your computer, then run `make evaluate`.

## 1. Settings
Pick the run. Change only these lines.

In [ ]:
RUN_NAME = "unet_28spk_psm"           # or "unet_28spk_c8"
EXTRA_ARGS = ["--target", "psm"]      # or ["--base-channels", "8"]

DRIVE_DIR = "/content/drive/MyDrive/noisy_speech_runs"   # checkpoints and outputs (survive a disconnect)
DATA_DIR = "/content/valentini"                          # dataset (Colab's disk; downloaded each session)
CACHE_DIR = "/content/cache"                             # 16 kHz cache (rebuilt each session)
REPO_URL = "https://github.com/jlwise0111/noisy_speech.git"
REPO_DIR = "/content/noisy_speech"

# The metric suite skips folders with "train" in the name, and the viewer files a run under
# "Neural network" only when its path contains "unet".
assert "train" not in RUN_NAME.replace("28spk", ""), "run names must not contain 'train'"
assert "unet" in RUN_NAME, "keep 'unet' in the run name so the viewer and metric suite find it"

## 2. Connect Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

## 3. Get the code and check the GPU

In [ ]:
import base64, subprocess, sys

def secret(name):
    """A value from Colab's Secrets panel, or None when it isn't set or notebook access is off."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

TOKEN = secret("GITHUB_TOKEN")
# The header git sends: the token in base64, which is just as secret as the token itself.
BASIC = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode() if TOKEN else None

def run(cmd):
    """Run a command and show its output. Colab doesn't display a subprocess's output on its own,
    so it is captured and printed, with the token blanked out. Stops the notebook on failure."""
    def hide(text):
        for value in (TOKEN, BASIC):
            if value:
                text = text.replace(value, "***")
        return text
    print("$", hide(" ".join(cmd)))
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = hide((result.stdout or "") + (result.stderr or "")).strip()
    if output:
        print(output)
    if result.returncode != 0:
        raise SystemExit(f"Command failed (exit code {result.returncode}); see the message above.")

def git(*args):
    """git, logged in with the token for this one command (nothing is written to disk)."""
    auth = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {BASIC}"] if TOKEN else []
    try:
        run(["git", *auth, *args])
    except SystemExit:
        if TOKEN:
            print("GitHub refused the token. Check it hasn't expired and that it has "
                  "Contents: Read-only access to the noisy_speech repository.")
        raise

if not TOKEN:
    print("No GITHUB_TOKEN secret found. Cloning will only work if the repository is public.")
if os.path.isdir(f"{REPO_DIR}/.git"):
    git("-C", REPO_DIR, "pull", "-q")
else:
    git("clone", "-q", REPO_URL, REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/unet_model/requirements.txt"])
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()

import torch
print(f"code at commit {commit}; torch {torch.__version__}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then run the cells again.")
print("GPU:", torch.cuda.get_device_name(0))

## 4. Download the dataset (about 5.6 GB, roughly 5–15 minutes)
Only the four folders this run needs: the 28-speaker training set and the test set.

In [ ]:
import time
from pathlib import Path

BASE_URL = "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/"
EXPECTED = {"clean_testset_wav": 824, "noisy_testset_wav": 824,
            "clean_trainset_28spk_wav": 11572, "noisy_trainset_28spk_wav": 11572}

def find_root():
    """The folder that holds all four dataset folders (unzipping may add a level), or None."""
    for d in [Path(DATA_DIR), *Path(DATA_DIR).glob("*")]:
        if all((d / name).is_dir() for name in EXPECTED):
            return d
    return None

def counts(root):
    return {name: sum(1 for p in (root / name).glob("*.wav") if not p.name.startswith("._"))
            for name in EXPECTED}

root = find_root()
if root is None or counts(root) != EXPECTED:
    os.makedirs(DATA_DIR, exist_ok=True)
    for name in EXPECTED:
        zip_path = f"/content/{name}.zip"
        # -nv prints one line per file and any server error; the retry flags ride out brief
        # server errors and rate limits; -c resumes a partial download.
        wget = ["wget", "-nv", "-c", "--tries=6", "--waitretry=30", "--timeout=60",
                "--retry-on-http-error=429,500,502,503,504", BASE_URL + name + ".zip", "-O", zip_path]
        try:
            run(wget)
        except SystemExit:
            # A leftover partial file can make the server refuse the resume; start that file over once.
            if os.path.exists(zip_path):
                os.remove(zip_path)
            print("Retrying from scratch in 60 s...")
            time.sleep(60)
            try:
                run(wget)
            except SystemExit:
                raise SystemExit("DataShare refused the download (see the message above). If it says 403 or 429, "
                                 "the server is limiting Colab: wait 15-30 minutes and run this cell again.")
        run(["unzip", "-q", "-o", zip_path, "-d", DATA_DIR])
        os.remove(zip_path)
    root = find_root()

if root is None:
    raise SystemExit(f"Dataset folders not found under {DATA_DIR}: check the unzip output above.")
found = counts(root)
for name, n in found.items():
    print(f"{name:26s} {n:6d} files" + ("" if n == EXPECTED[name] else f"   <- expected {EXPECTED[name]}"))
if found != EXPECTED:
    raise SystemExit("File counts don't match; delete the dataset folder and run this cell again.")
DATA_ROOT = root
print("dataset:", DATA_ROOT)

## 5. Train and enhance the test set

The first lines should show validation speakers **p256, p279, p287** and `device cuda`. Building the 16 kHz
cache takes a few minutes before epoch 1. `--workers 4` keeps the training crops identical to the main
run; Colab has only 2 CPU cores, so PyTorch may warn that 4 workers is more than suggested. That's expected.

In [ ]:
run_dir = Path(DRIVE_DIR) / RUN_NAME
config = run_dir / "config.json"
log_path = Path(DRIVE_DIR) / f"{RUN_NAME}.log"

if config.exists() and '"seconds"' in config.read_text():
    print(f"{RUN_NAME} already finished; nothing to do. (Delete {run_dir} on Drive to train it again.)")
else:
    cmd = [sys.executable, "-u", "train_unet.py", "--dataset", str(DATA_ROOT), "--run-name", RUN_NAME,
           "--workers", "4", "--out", DRIVE_DIR, "--cache-dir", CACHE_DIR, *EXTRA_ARGS]
    if (run_dir / "last.pt").exists():
        cmd.append("--resume")        # continue after a disconnect
    print("$", " ".join(cmd))
    with open(log_path, "a") as log:
        proc = subprocess.Popen(cmd, cwd=f"{REPO_DIR}/unet_model", stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
    if proc.wait() != 0:
        raise SystemExit(f"Training stopped with an error (exit code {proc.returncode}); see the output above.")

## 6. Check the result

In [ ]:
import json

info = json.loads(config.read_text())
if "seconds" not in info:
    raise SystemExit("The run hasn't finished yet: run the training cell again (it resumes).")
print(f"{RUN_NAME}: best epoch {info['best_epoch']}, val loss {info['val_loss']:.5f}, "
      f"{info['n_test']} test files enhanced in {info['seconds'] / 3600:.1f} h (commit {info['git_commit'][:7]})")
print(f"Next: download {run_dir} from Google Drive into unet_model/outputs/{RUN_NAME}/ and run `make evaluate`.")